# Dataset ficticio de mantenimiento industrial — generación de datos

Este notebook genera un conjunto de tablas de forma artificial y estadísticamente factible,
provenientes de una base de datos de un sistema de gestión y monitoreo de mantenimiento industrial para
**5 plantas** con activos comparables, a lo largo de un período de
**2 años**.

El objetivo es simular los archivos CSV que entregaría un equipo de ingeniería de datos, para
luego trabajar el análisis un notebook
separado.

**Orden de generación** (respeta las dependencias por clave foránea):

1. Catálogos puros (`tipo_activo`, `tipo_falla`, `tipo_medicion`,
   `tipo_evento_operativo`, `repuestos_insumos`, `tipo_cambio`)
2. `plantas`
3. `activos` (jerarquía recursiva)
4. `tecnicos`
5. `plan_mantenimiento_preventivo` y `ordenes`
6. `eventos_mantenimiento`
7. `evento_tecnico`, `evento_repuesto`
8. `mediciones` (horas de uso, con Weibull) y `log_operativo`




## 0. Configuración general

In [2]:
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
import json
import os

# Semilla fija: todo el notebook es reproducible de punta a punta.
# Usamos un Generator moderno de numpy en vez de np.random.seed clásico.
rng = np.random.default_rng(42)

# Ventana temporal del dataset: 2 años de historia
FECHA_INICIO = date(2024, 1, 1)
FECHA_FIN    = date(2025, 12, 31)

# Carpeta de salida para los CSV "entregados"
OUT_DIR = "output_csv"
os.makedirs(OUT_DIR, exist_ok=True)

def guardar(df, nombre):
    """Guarda un DataFrame como CSV en la carpeta de salida y
    muestra un resumen rápido (shape + primeras filas) para
    verificar que la generación tenga sentido antes de seguir."""
    path = os.path.join(OUT_DIR, f"{nombre}.csv")
    df.to_csv(path, index=False)
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas -> {path}")
    #return df


## 1. Tablas puras

Tablas sin dependencias (no tienen FK hacia otras tablas del modelo).
Se generan primero porque el resto del dataset las referencia.


### 1.1 `tipo_activo`

Catálogo de tipos de equipo, con una columna `categoria` para poder agrupar en el análisis (eléctrico / mecánico / hidráulico / combustión). Se incluye también `Área/Sector`, que es el tipo que usamos para los nodos contenedores de nivel 1 en la jerarquía de `activos` (no es un equipo físico, es una posición organizativa).

In [3]:
tipo_activo_data = [
    ("Área/Sector",                    "Estructural"),
    ("Bomba centrífuga",               "Hidráulico/Mecánico"),
    ("Bomba hidráulica",               "Hidráulico"),
    ("Motor eléctrico CC",             "Eléctrico"),
    ("Motor eléctrico AC",             "Eléctrico"),
    ("Motor diesel",                   "Combustión"),
    ("Motor hidráulico",               "Hidráulico"),
    ("Transformador",                  "Eléctrico"),
    ("Generador eléctrico",            "Eléctrico"),
    ("Pistón hidráulico",              "Hidráulico"),
    ("Compresor de tornillo",          "Mecánico"),
    ("Compresor a pistón",             "Mecánico"),
    ("Cinta transportadora",           "Mecánico"),
    ("Variador de frecuencia",         "Eléctrico/Electrónico"),
    ("Tablero distribución eléctrica", "Eléctrico"),
    ("Tablero de control",             "Eléctrico/Electrónico"),
    ("Electroválvula",                 "Eléctrico/Mecánico"),
    ("Válvula esclusa",                "Mecánico"),
    ("Puente grúa",                    "Mecánico"),
    ("Malacate",                       "Mecánico"),
]
tipo_activo = pd.DataFrame(tipo_activo_data, columns=["nombre", "categoria"])
tipo_activo.insert(0, "tipo_activo_id", range(1, len(tipo_activo) + 1))

# lookup nombre -> id, muy usado más adelante al armar activos/planes
TA = dict(zip(tipo_activo["nombre"], tipo_activo["tipo_activo_id"]))

guardar(tipo_activo, "tipo_activo")
tipo_activo.head()

tipo_activo: 20 filas, 3 columnas -> output_csv/tipo_activo.csv


,tipo_activo_id,nombre,categoria
0,1,Área/Sector,Estructural
1,2,Bomba centrífuga,Hidráulico/Mecánico
2,3,Bomba hidráulica,Hidráulico
3,4,Motor eléctrico CC,Eléctrico
4,5,Motor eléctrico AC,Eléctrico


### 1.2 `tipo_falla`

Incluye causas genéricas y algunas específicas por familia de activo, más el ítem especial `Mantenimiento programado (sin falla)` para eventos preventivos de rutina. Recordá que `tipo_falla_id` va a quedar nullable en `eventos_mantenimiento`, para poder distinguir después entre "no aplica (preventivo)" y "correctivo sin registrar" (dato faltante real).

In [3]:
tipo_falla_data = [
    "Desgaste mecánico",
    "Sobrecalentamiento",
    "Cortocircuito",
    "Fuga de fluido",
    "Vibración excesiva",
    "Falla de aislación",
    "Obstrucción / atascamiento",
    "Corrosión",
    "Falla de lubricación",
    "Falla de rodamiento",
    "Error de programación/control",
    "Rotura estructural",
    "Fin de vida útil",
    "Mantenimiento programado (sin falla)",
]
tipo_falla = pd.DataFrame({"descripcion": tipo_falla_data})
tipo_falla.insert(0, "tipo_falla_id", range(1, len(tipo_falla) + 1))
TF = dict(zip(tipo_falla["descripcion"], tipo_falla["tipo_falla_id"]))

guardar(tipo_falla, "tipo_falla")
tipo_falla

tipo_falla: 14 filas, 2 columnas -> output_csv/tipo_falla.csv


,tipo_falla_id,descripcion
0,1,Desgaste mecánico
1,2,Sobrecalentamiento
2,3,Cortocircuito
3,4,Fuga de fluido
4,5,Vibración excesiva
5,6,Falla de aislación
6,7,Obstrucción / atascamiento
7,8,Corrosión
8,9,Falla de lubricación
9,10,Falla de rodamiento


### 1.3 `tipo_medicion`

Por decisión de diseño, el modelo queda genérico (EAV) aunque en esta primera etapa solo carguemos horas de uso acumuladas. Deja la puerta abierta a sumar temperatura/vibración más adelante sin tocar el esquema.

In [4]:
tipo_medicion_data = ["Horas de uso acumuladas"]
tipo_medicion = pd.DataFrame({"nombre": tipo_medicion_data})
tipo_medicion.insert(0, "tipo_medicion_id", range(1, len(tipo_medicion) + 1))
TM = dict(zip(tipo_medicion["nombre"], tipo_medicion["tipo_medicion_id"]))

guardar(tipo_medicion, "tipo_medicion")
tipo_medicion

tipo_medicion: 1 filas, 2 columnas -> output_csv/tipo_medicion.csv


,tipo_medicion_id,nombre
0,1,Horas de uso acumuladas


### 1.4 `tipo_evento_operativo`

A propósito, ruidoso: mezcla mayúsculas/minúsculas y variantes del mismo concepto (parada / PARADA / STOP), tal como aparecería en un log real de SCADA/historian con carga mixta humana + automática. Este "ensuciado" intencional es la base de un ejercicio de limpieza y normalización de texto en la etapa de análisis.

In [5]:
tipo_evento_operativo_data = [
    "PARADA", "parada_planta", "STOP", "Parada por falla", "parada prog.",
    "ARRANQUE", "arranque_planta", "START", "puesta en marcha",
    "drenaje", "DRENAJE LN2", "purga", "PURGA VALVULA", "limpieza filtros",
    "ALARMA TEMP", "disparo_termico", "proteccion activada", "ALARM_HIGH_PRESS",
    "cambio_turno", "CAMBIO DE TURNO", "ajuste_setpoint", "cambio de producto",
    "corte_suministro", "CORTE ELECTRICO EXTERNO", "falta de agua", "corte gas",
    "producto_fuera_especificacion", "PARO POR CALIDAD", "merma",
]
tipo_evento_operativo = pd.DataFrame({"nombre": tipo_evento_operativo_data})
tipo_evento_operativo.insert(0, "tipo_evento_id", range(1, len(tipo_evento_operativo) + 1))
TEO = dict(zip(tipo_evento_operativo["nombre"], tipo_evento_operativo["tipo_evento_id"]))

guardar(tipo_evento_operativo, "tipo_evento_operativo")
tipo_evento_operativo

tipo_evento_operativo: 29 filas, 2 columnas -> output_csv/tipo_evento_operativo.csv


,tipo_evento_id,nombre
0,1,PARADA
1,2,parada_planta
2,3,STOP
3,4,Parada por falla
4,5,parada prog.
5,6,ARRANQUE
6,7,arranque_planta
7,8,START
8,9,puesta en marcha
9,10,drenaje


### 1.5 `repuestos_insumos`

Catálogo con `costo_unitario` como referencia "actual/lista" — el costo real pagado en cada evento se va a guardar aparte, en la tabla puente `evento_repuesto`, para preservar el precio histórico.

In [6]:
repuestos_data = [
    ("Rodamiento estándar",        "repuesto", "unidad", 45000),
    ("Aceite dieléctrico",         "insumo",   "litro",  3200),
    ("Aceite lubricante industrial","insumo",  "litro",  2800),
    ("Correa transportadora",      "repuesto", "unidad", 180000),
    ("Sello mecánico",             "repuesto", "unidad", 32000),
    ("Filtro de aire",             "insumo",   "unidad", 8500),
    ("Filtro de aceite",           "insumo",   "unidad", 9500),
    ("Fusible de potencia",        "repuesto", "unidad", 15000),
    ("Contactor eléctrico",        "repuesto", "unidad", 60000),
    ("Bobinado de repuesto",       "repuesto", "unidad", 250000),
    ("Grasa industrial",           "insumo",   "kg",     4200),
    ("Manguera hidráulica",        "repuesto", "metro",  12000),
    ("Válvula de repuesto",        "repuesto", "unidad", 95000),
    ("Kit de sellos hidráulicos",  "repuesto", "unidad", 55000),
    ("Cable eléctrico",            "insumo",   "metro",  2500),
]
repuestos_insumos = pd.DataFrame(
    repuestos_data, columns=["nombre", "tipo", "unidad_medida", "costo_unitario"]
)
repuestos_insumos.insert(0, "repuesto_id", range(1, len(repuestos_insumos) + 1))
REP = dict(zip(repuestos_insumos["nombre"], repuestos_insumos["repuesto_id"]))

guardar(repuestos_insumos, "repuestos_insumos")
repuestos_insumos

repuestos_insumos: 15 filas, 5 columnas -> output_csv/repuestos_insumos.csv


,repuesto_id,nombre,tipo,unidad_medida,costo_unitario
0,1,Rodamiento estándar,repuesto,unidad,45000
1,2,Aceite dieléctrico,insumo,litro,3200
2,3,Aceite lubricante industrial,insumo,litro,2800
3,4,Correa transportadora,repuesto,unidad,180000
4,5,Sello mecánico,repuesto,unidad,32000
5,6,Filtro de aire,insumo,unidad,8500
6,7,Filtro de aceite,insumo,unidad,9500
7,8,Fusible de potencia,repuesto,unidad,15000
8,9,Contactor eléctrico,repuesto,unidad,60000
9,10,Bobinado de repuesto,repuesto,unidad,250000


### 1.6 `tipo_cambio`

Un valor promedio de tipo de cambio (ARS/USD) por mes, para los 24 meses del período. Simulamos una trayectoria con inflación mensual variable (ruido alrededor de una tendencia), no un crecimiento perfectamente lineal — así el ejercicio de conversión a USD en el análisis es más realista.

In [7]:
meses = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="MS")  # inicio de cada mes

# tasa de devaluación mensual: media ~4% con variabilidad, valor inicial ~900 ARS/USD
tasa_mensual = rng.normal(loc=0.04, scale=0.015, size=len(meses))
valores = [900.0]
for tasa in tasa_mensual[1:]:
    valores.append(valores[-1] * (1 + tasa))

tipo_cambio = pd.DataFrame({
    "periodo": meses,
    "valor_promedio": np.round(valores, 2),
})
tipo_cambio.insert(0, "tipo_cambio_id", range(1, len(tipo_cambio) + 1))

guardar(tipo_cambio, "tipo_cambio")
tipo_cambio.head()

tipo_cambio: 24 filas, 3 columnas -> output_csv/tipo_cambio.csv


,tipo_cambio_id,periodo,valor_promedio
0,1,2024-01-01,900.00
1,2,2024-02-01,921.96
2,3,2024-03-01,969.22
3,4,2024-04-01,1021.66
4,5,2024-05-01,1032.63


## 2. `plantas`

5 plantas con ubicación y zona climática distintas, para poder comparar tendencias entre entornos más adelante (una de las motivaciones centrales del proyecto).

In [8]:
plantas_data = [
    ("Planta Norte",      "Salta",         "Cálido seco",     date(2018, 3, 1)),
    ("Planta Litoral",    "Santa Fe",      "Templado húmedo", date(2016, 6, 15)),
    ("Planta Patagonia",  "Neuquén",       "Frío árido",      date(2019, 11, 1)),
    ("Planta Centro",     "Córdoba",       "Templado",        date(2015, 1, 20)),
    ("Planta Bonaerense", "Buenos Aires",  "Templado húmedo", date(2020, 2, 10)),
]
plantas = pd.DataFrame(
    plantas_data,
    columns=["nombre", "ubicacion", "zona_climatica", "fecha_puesta_marcha"],
)
plantas.insert(0, "planta_id", range(1, len(plantas) + 1))

guardar(plantas, "plantas")
plantas

plantas: 5 filas, 5 columnas -> output_csv/plantas.csv


,planta_id,nombre,ubicacion,zona_climatica,fecha_puesta_marcha
0,1,Planta Norte,Salta,Cálido seco,2018-03-01
1,2,Planta Litoral,Santa Fe,Templado húmedo,2016-06-15
2,3,Planta Patagonia,Neuquén,Frío árido,2019-11-01
3,4,Planta Centro,Córdoba,Templado,2015-01-20
4,5,Planta Bonaerense,Buenos Aires,Templado húmedo,2020-02-10


## 3. `activos` — jerarquía recursiva

Estructura de 3 niveles:

- **Nivel 1**: áreas/sectores dentro de la planta (nodos contenedores,
  `tipo_activo = "Área/Sector"`, sin características técnicas).
- **Nivel 2**: equipos "mayores" dentro de cada área (transformadores,
  bombas, motores...), tomados del catálogo `tipo_activo` según un
  mapeo laxo de qué equipos son razonables en cada tipo de área.
- **Nivel 3**: sub-componentes de *algunos* equipos (no todos), por
  ejemplo el malacate de un puente grúa, o el motor que impulsa una
  bomba centrífuga — con probabilidad, no determinístico.

Cada fila lleva `planta_id` desnormalizado (aunque se pueda inferir
subiendo por `padre_id`), `posicion_relativa` como texto libre "sucio"
(simulando data entry real), y `caracteristicas` como JSON con solo
constantes del equipo (nunca variables dinámicas como horas de uso).


In [9]:
AREAS_POSIBLES = [
    "Sala de máquinas", "Subestación eléctrica", "Línea de producción 1",
    "Línea de producción 2", "Sala de compresores", "Depósito de repuestos",
    "Sala de bombeo", "Playa de maniobras",
]

# qué tipos de equipo tiene sentido encontrar dentro de cada tipo de área
EQUIPOS_POR_AREA = {
    "Sala de máquinas":      ["Motor eléctrico AC", "Motor eléctrico CC", "Motor diesel", "Generador eléctrico"],
    "Subestación eléctrica": ["Transformador", "Tablero distribución eléctrica", "Generador eléctrico", "Variador de frecuencia"],
    "Línea de producción 1": ["Cinta transportadora", "Motor eléctrico AC", "Variador de frecuencia", "Electroválvula", "Tablero de control"],
    "Línea de producción 2": ["Cinta transportadora", "Motor eléctrico AC", "Variador de frecuencia", "Electroválvula", "Puente grúa"],
    "Sala de compresores":   ["Compresor de tornillo", "Compresor a pistón", "Motor eléctrico AC", "Tablero de control"],
    "Depósito de repuestos": ["Puente grúa", "Malacate"],
    "Sala de bombeo":        ["Bomba centrífuga", "Bomba hidráulica", "Motor eléctrico AC", "Motor eléctrico CC", "Válvula esclusa", "Electroválvula"],
    "Playa de maniobras":    ["Puente grúa", "Malacate", "Motor hidráulico", "Pistón hidráulico"],
}

# equipos "mayores" que pueden tener sub-componentes propios (nivel 3)
SUBCOMPONENTES = {
    "Puente grúa":           ["Malacate"],
    "Tablero de control":    ["Variador de frecuencia"],
    "Bomba centrífuga":      ["Motor eléctrico AC"],
    "Compresor de tornillo": ["Motor eléctrico AC"],
}

# plantillas de posición "sucia": texto libre, inconsistente a propósito
POSICION_TEMPLATES = [
    "al lado del tablero ppal", "Rack {n}", "cerca ventana norte",
    "jaula {n}", "PASILLO {n}", "fondo del galpón", "al lado de la puerta",
    "sector {n} izq", "junto a bomba anterior", "esquina NE", "s/d",
    "detrás del compresor", "nivel superior, sala {n}", "entrada principal",
]

CRITICIDADES = ["Alta", "Media", "Baja"]


def texto_posicion(rng):
    return rng.choice(POSICION_TEMPLATES).format(n=rng.integers(1, 6))


def numero_serie(rng, tipo_nombre):
    prefijo = "".join(w[0] for w in tipo_nombre.split())[:3].upper()
    return f"{prefijo}-{rng.integers(10000, 99999)}"


def caracteristicas_json(rng, tipo_nombre, categoria):
    """
    Dict de características CONSTANTES del activo (nunca series
    temporales). El contenido depende de la categoría del tipo de equipo.
    Se guarda luego como JSON en la columna `caracteristicas`.
    """
    car = {}
    if tipo_nombre == "Área/Sector":
        return car  # las áreas no tienen características técnicas

    if "Eléctrico" in categoria or tipo_nombre in ("Transformador", "Generador eléctrico"):
        car["voltaje_v"] = int(rng.choice([220, 380, 660, 6600, 13200]))
        car["potencia_kw"] = round(float(rng.uniform(5, 500)), 1)
    if "Hidráulico" in categoria:
        car["caudal_m3h"] = round(float(rng.uniform(2, 80)), 1)
        car["presion_bar"] = round(float(rng.uniform(10, 250)), 1)
    if tipo_nombre == "Motor diesel":
        car["potencia_hp"] = round(float(rng.uniform(50, 400)), 1)
        car["combustible"] = "diesel"

    if car:
        car["modelo"] = f"MOD-{rng.integers(100, 999)}"
        car["numero_parte"] = f"NP{rng.integers(10000, 99999)}"
    return car


def generar_activos_planta(planta_id, fecha_puesta_marcha, id_counter, rng):
    """
    Genera la jerarquía de activos (niveles 1, 2 y 3) para UNA planta.
    `id_counter` es una lista de un elemento usada como contador global
    mutable, para que los activo_id sean únicos entre plantas.
    """
    filas = []
    n_areas = rng.integers(4, 7)  # entre 4 y 6 áreas por planta
    areas_elegidas = rng.choice(AREAS_POSIBLES, size=n_areas, replace=False)

    for area_nombre in areas_elegidas:
        area_id = id_counter[0]; id_counter[0] += 1
        filas.append(dict(
            activo_id=area_id, padre_id=None, planta_id=planta_id,
            tipo_activo_id=TA["Área/Sector"], codigo=f"AREA-{area_id}",
            nivel_jerarquico=1, posicion_relativa="N/A (nodo de área)",
            numero_serie=None, reemplaza_a=None,
            criticidad="Media", fecha_alta=fecha_puesta_marcha,
            estado="activo", caracteristicas=json.dumps({}, ensure_ascii=False),
        ))

        equipos_posibles = EQUIPOS_POR_AREA[area_nombre]
        n_equipos = rng.integers(3, 8)
        for _ in range(n_equipos):
            tipo_nombre = rng.choice(equipos_posibles)
            categoria = tipo_activo.loc[tipo_activo.nombre == tipo_nombre, "categoria"].iloc[0]
            eq_id = id_counter[0]; id_counter[0] += 1
            filas.append(dict(
                activo_id=eq_id, padre_id=area_id, planta_id=planta_id,
                tipo_activo_id=TA[tipo_nombre], codigo=f"EQ-{eq_id}",
                nivel_jerarquico=2, posicion_relativa=texto_posicion(rng),
                numero_serie=numero_serie(rng, tipo_nombre),
                reemplaza_a=None,
                criticidad=rng.choice(CRITICIDADES, p=[0.3, 0.45, 0.25]),
                fecha_alta=fecha_puesta_marcha,
                estado="activo",
                caracteristicas=json.dumps(
                    caracteristicas_json(rng, tipo_nombre, categoria), ensure_ascii=False
                ),
            ))

            # nivel 3: sub-componentes, solo para algunos tipos y con
            # probabilidad (no todos los equipos de ese tipo tienen hijos)
            if tipo_nombre in SUBCOMPONENTES and rng.random() < 0.35:
                for sub_nombre in SUBCOMPONENTES[tipo_nombre]:
                    sub_categoria = tipo_activo.loc[tipo_activo.nombre == sub_nombre, "categoria"].iloc[0]
                    sub_id = id_counter[0]; id_counter[0] += 1
                    filas.append(dict(
                        activo_id=sub_id, padre_id=eq_id, planta_id=planta_id,
                        tipo_activo_id=TA[sub_nombre], codigo=f"SUB-{sub_id}",
                        nivel_jerarquico=3, posicion_relativa=texto_posicion(rng),
                        numero_serie=numero_serie(rng, sub_nombre),
                        reemplaza_a=None,
                        criticidad=rng.choice(CRITICIDADES, p=[0.2, 0.5, 0.3]),
                        fecha_alta=fecha_puesta_marcha,
                        estado="activo",
                        caracteristicas=json.dumps(
                            caracteristicas_json(rng, sub_nombre, sub_categoria), ensure_ascii=False
                        ),
                    ))
    return filas


id_counter = [1]
todas_filas = []
for _, planta in plantas.iterrows():
    todas_filas.extend(
        generar_activos_planta(planta.planta_id, planta.fecha_puesta_marcha, id_counter, rng)
    )

activos = pd.DataFrame(todas_filas)
guardar(activos, "activos")
activos.head(10)

activos: 146 filas, 13 columnas -> output_csv/activos.csv


,activo_id,padre_id,planta_id,tipo_activo_id,codigo,nivel_jerarquico,posicion_relativa,numero_serie,reemplaza_a,criticidad,fecha_alta,estado,caracteristicas
0,1,NaN,1,1,AREA-1,1,N/A (nodo de área),NaN,None,Media,2018-03-01,activo,{}
1,2,1.0,1,20,EQ-2,2,fondo del galpón,M-46976,None,Media,2018-03-01,activo,{}
2,3,1.0,1,19,EQ-3,2,Rack 3,PG-81610,None,Alta,2018-03-01,activo,{}
3,4,1.0,1,19,EQ-4,2,PASILLO 2,PG-60804,None,Media,2018-03-01,activo,{}
4,5,1.0,1,20,EQ-5,2,junto a bomba anterior,M-18750,None,Baja,2018-03-01,activo,{}
5,6,1.0,1,19,EQ-6,2,al lado de la puerta,PG-85751,None,Alta,2018-03-01,activo,{}
6,7,1.0,1,19,EQ-7,2,junto a bomba anterior,PG-84945,None,Alta,2018-03-01,activo,{}
7,8,1.0,1,19,EQ-8,2,s/d,PG-52440,None,Baja,2018-03-01,activo,{}
8,9,NaN,1,1,AREA-9,1,N/A (nodo de área),NaN,None,Media,2018-03-01,activo,{}
9,10,9.0,1,14,EQ-10,2,sector 1 izq,VDF-22581,None,Alta,2018-03-01,activo,"{""voltaje_v"": 660, ""potencia_kw"": 238.2, ""mode..."


### 3.1 Verificación rápida

Antes de seguir con técnicos y eventos, conviene chequear que la jerarquía tenga sentido: cantidad de activos por nivel y por planta, y que no haya huérfanos (padre_id que no exista).

In [10]:
print("Activos por nivel jerárquico:")
print(activos.nivel_jerarquico.value_counts().sort_index())
print()
print("Activos por planta:")
print(activos.planta_id.value_counts().sort_index())
print()

# chequeo de integridad: todo padre_id no nulo debe existir como activo_id
ids_validos = set(activos.activo_id)
padres_invalidos = activos[
    activos.padre_id.notna() & ~activos.padre_id.isin(ids_validos)
]
print(f"Filas con padre_id inválido: {len(padres_invalidos)} (debería ser 0)")

Activos por nivel jerárquico:


nivel_jerarquico
1     24
2    116
3      6
Name: count, dtype: int64

Activos por planta:
planta_id
1    37
2    24
3    20
4    31
5    34
Name: count, dtype: int64

Filas con padre_id inválido: 0 (debería ser 0)


## 4. `tecnicos`

Entre 6 y 10 técnicos por planta, fijos (sin rotación entre plantas,
según lo acordado). Usamos `Faker` con locale `es_AR` para nombres
realistas.


In [11]:
from faker import Faker

faker = Faker("es_AR")
Faker.seed(42)  # reproducibilidad también para Faker

ESPECIALIDADES = ["Eléctrico", "Mecánico", "Hidráulico", "Electrónico/Control", "General"]

filas_tec = []
tecnico_id = 1
for _, planta in plantas.iterrows():
    n_tecnicos = rng.integers(6, 11)  # entre 6 y 10 técnicos por planta
    for _ in range(n_tecnicos):
        fecha_ingreso = faker.date_between(start_date=date(2010, 1, 1), end_date=FECHA_INICIO)
        filas_tec.append(dict(
            tecnico_id=tecnico_id,
            planta_id=planta.planta_id,
            nombre=faker.name(),
            especialidad=rng.choice(ESPECIALIDADES, p=[0.3, 0.3, 0.15, 0.15, 0.10]),
            fecha_ingreso=fecha_ingreso,
            estado="activo",
        ))
        tecnico_id += 1

tecnicos = pd.DataFrame(filas_tec)
guardar(tecnicos, "tecnicos")
tecnicos.head()

tecnicos: 37 filas, 6 columnas -> output_csv/tecnicos.csv


,tecnico_id,planta_id,nombre,especialidad,fecha_ingreso,estado
0,1,1,Thiago Nicolas Martinez Fernandez,Electrónico/Control,2018-12-14,activo
1,2,1,Sr(a). Salvador Gonzalez,Eléctrico,2011-06-09,activo
2,3,1,Juan Bautista Martina Diaz,Eléctrico,2011-04-25,activo
3,4,1,Dr(a). Delfina Nuñez,Mecánico,2019-02-05,activo
4,5,1,Alejo Fernandez Flores,Hidráulico,2021-05-01,activo


## 5. `plan_mantenimiento_preventivo` y `ordenes`

### 5.1 El plan (la plantilla)

Definimos tareas típicas por tipo de activo, con su frecuencia. No
todos los tipos tienen plan (las áreas no llevan mantenimiento propio,
son solo nodos organizativos). Las frecuencias mezclan varios tipos
(`semanal`, `mensual`, `anual`, `por_horas_uso`, `por_parada_planta`)
a propósito, para que la generación de órdenes tenga que resolver
cada caso de forma distinta — igual que en un CMMS real.


In [12]:
PLANES_POR_TIPO = {
    "Transformador":                  [("Análisis de aceite dieléctrico", "anual", 1, 30),
                                        ("Inspección termográfica", "mensual", 3, 15)],
    "Generador eléctrico":            [("Cambio de aceite y filtros", "por_horas_uso", 500, 15),
                                        ("Prueba de arranque en vacío", "mensual", 1, 7)],
    "Motor eléctrico AC":             [("Lubricación de rodamientos", "por_horas_uso", 2000, 15),
                                        ("Medición de vibración", "mensual", 1, 10)],
    "Motor eléctrico CC":             [("Inspección de escobillas", "mensual", 2, 10)],
    "Motor diesel":                   [("Cambio de aceite", "por_horas_uso", 250, 10),
                                        ("Cambio de filtros", "por_horas_uso", 500, 10)],
    "Motor hidráulico":               [("Inspección de sellos", "mensual", 2, 15)],
    "Bomba centrífuga":               [("Lubricación y alineación", "por_horas_uso", 1500, 15)],
    "Bomba hidráulica":               [("Cambio de aceite hidráulico", "por_horas_uso", 1000, 15)],
    "Pistón hidráulico":              [("Inspección de fugas", "mensual", 1, 10)],
    "Compresor de tornillo":          [("Cambio de filtros de aire", "por_horas_uso", 1000, 10),
                                        ("Purga de condensado", "semanal", 1, 5)],
    "Compresor a pistón":             [("Cambio de aceite", "por_horas_uso", 800, 10)],
    "Cinta transportadora":           [("Inspección de correa y rodillos", "semanal", 1, 5)],
    "Variador de frecuencia":         [("Limpieza de disipadores", "mensual", 3, 15)],
    "Tablero distribución eléctrica": [("Termografía y ajuste de bornes", "anual", 1, 30)],
    "Tablero de control":             [("Backup de programa y calibración", "anual", 1, 20)],
    "Electroválvula":                 [("Prueba de apertura/cierre", "mensual", 2, 10)],
    "Válvula esclusa":                [("Lubricación de vástago", "mensual", 2, 10)],
    "Puente grúa":                    [("Inspección de cables y frenos", "mensual", 1, 7),
                                        ("Parada de planta - revisión mayor", "por_parada_planta", 1, 10)],
    "Malacate":                       [("Inspección de cable y gancho", "mensual", 1, 7)],
}

filas_plan = []
plan_id = 1
for tipo_nombre, tareas in PLANES_POR_TIPO.items():
    for descripcion, freq_tipo, freq_valor, plazo in tareas:
        filas_plan.append(dict(
            plan_id=plan_id,
            tipo_activo_id=TA[tipo_nombre],
            descripcion_tarea=descripcion,
            frecuencia_tipo=freq_tipo,
            frecuencia_valor=freq_valor,
            plazo_cumplimiento_dias=plazo,
        ))
        plan_id += 1

plan_mantenimiento_preventivo = pd.DataFrame(filas_plan)
guardar(plan_mantenimiento_preventivo, "plan_mantenimiento_preventivo")
plan_mantenimiento_preventivo.head(10)

plan_mantenimiento_preventivo: 25 filas, 6 columnas -> output_csv/plan_mantenimiento_preventivo.csv


,plan_id,tipo_activo_id,descripcion_tarea,frecuencia_tipo,frecuencia_valor,plazo_cumplimiento_dias
0,1,8,Análisis de aceite dieléctrico,anual,1,30
1,2,8,Inspección termográfica,mensual,3,15
2,3,9,Cambio de aceite y filtros,por_horas_uso,500,15
3,4,9,Prueba de arranque en vacío,mensual,1,7
4,5,5,Lubricación de rodamientos,por_horas_uso,2000,15
5,6,5,Medición de vibración,mensual,1,10
6,7,4,Inspección de escobillas,mensual,2,10
7,8,6,Cambio de aceite,por_horas_uso,250,10
8,9,6,Cambio de filtros,por_horas_uso,500,10
9,10,7,Inspección de sellos,mensual,2,15


### 5.2 Paradas de planta programadas (a mano)

Tal como acordamos, esto **no forma parte del modelo relacional**: es
solo una referencia auxiliar para poder calcular el vencimiento de las
órdenes con `frecuencia_tipo = 'por_parada_planta'`. Cada planta para
2 veces por año, con un desfasaje propio para que no coincidan todas
en el mismo mes (más realista: no tiene sentido que las 5 plantas
frenen la misma semana).


In [13]:
PARADAS_PROGRAMADAS = {}
for _, planta in plantas.iterrows():
    offset = int(planta.planta_id) % 3
    mes1 = 4 + offset
    mes2 = (10 + offset - 1) % 12 + 1  # wrap simple a 1-12
    fechas = [
        date(2024, mes1, 15), date(2024, mes2, 15),
        date(2025, mes1, 15), date(2025, mes2, 15),
    ]
    PARADAS_PROGRAMADAS[planta.planta_id] = sorted(fechas)

for pid, fechas in PARADAS_PROGRAMADAS.items():
    print(f"Planta {pid}: {fechas}")

Planta 1: [datetime.date(2024, 5, 15), datetime.date(2024, 11, 15), datetime.date(2025, 5, 15), datetime.date(2025, 11, 15)]
Planta 2: [datetime.date(2024, 6, 15), datetime.date(2024, 12, 15), datetime.date(2025, 6, 15), datetime.date(2025, 12, 15)]
Planta 3: [datetime.date(2024, 4, 15), datetime.date(2024, 10, 15), datetime.date(2025, 4, 15), datetime.date(2025, 10, 15)]
Planta 4: [datetime.date(2024, 5, 15), datetime.date(2024, 11, 15), datetime.date(2025, 5, 15), datetime.date(2025, 11, 15)]
Planta 5: [datetime.date(2024, 6, 15), datetime.date(2024, 12, 15), datetime.date(2025, 6, 15), datetime.date(2025, 12, 15)]


### 5.3 `ordenes`: expandir el plan en instancias concretas

Para cada plan, recorremos todos los activos de ese tipo y generamos
las órdenes que hubieran salido en la ventana de 2 años, según su
frecuencia:

- **semanal/mensual/anual**: cadencia fija en días, con un desfasaje
  inicial aleatorio por activo (para que no todos los equipos del
  mismo tipo tengan la orden exactamente el mismo día).
- **por_horas_uso**: la convertimos a una cadencia aproximada en días
  asumiendo un promedio de uso diario. Es una simplificación
  deliberada — el detalle fino de horas reales lo resuelve más
  adelante la tabla `mediciones` con Weibull, pero para *emitir* la
  orden preventiva alcanza con una aproximación razonable.
- **por_parada_planta**: una orden por cada parada programada de esa
  planta, con vencimiento en la fecha de la parada.

También sumamos un puñado de **órdenes ad-hoc** (sin `plan_id`),
decididas por supervisión sin depender de un plan calendario, tal
como acordamos.

Dejamos `fecha_ejecucion` en `NaN` y `estado = "pendiente"` en todos
los casos: se resuelven recién en la próxima sección, cuando
generemos `eventos_mantenimiento` (una orden pasa a "cumplida" cuando
efectivamente existe un evento que la ejecuta).


In [14]:
HORAS_USO_PROMEDIO_DIA = 16  # simplificación para pasar horas -> días

filas_ordenes = []
orden_id = 1

activos_relevantes = activos[activos.nivel_jerarquico.isin([2, 3])]

for _, plan in plan_mantenimiento_preventivo.iterrows():
    activos_del_tipo = activos_relevantes[activos_relevantes.tipo_activo_id == plan.tipo_activo_id]

    for _, act in activos_del_tipo.iterrows():
        planta_id = act.planta_id

        if plan.frecuencia_tipo == "semanal":
            intervalo_dias = 7 * plan.frecuencia_valor
        elif plan.frecuencia_tipo == "mensual":
            intervalo_dias = 30 * plan.frecuencia_valor
        elif plan.frecuencia_tipo == "anual":
            intervalo_dias = 365 * plan.frecuencia_valor
        elif plan.frecuencia_tipo == "por_horas_uso":
            intervalo_dias = max(7, round(plan.frecuencia_valor / HORAS_USO_PROMEDIO_DIA))
        else:  # por_parada_planta
            intervalo_dias = None

        if plan.frecuencia_tipo == "por_parada_planta":
            for fecha_parada in PARADAS_PROGRAMADAS[planta_id]:
                fecha_emision = fecha_parada - timedelta(days=int(plan.plazo_cumplimiento_dias))
                if fecha_emision < FECHA_INICIO:
                    continue
                filas_ordenes.append(dict(
                    orden_id=orden_id, plan_id=plan.plan_id, activo_id=act.activo_id,
                    fecha_emision=fecha_emision, fecha_vencimiento=fecha_parada,
                    fecha_ejecucion=None, estado="pendiente",
                ))
                orden_id += 1
        else:
            offset_inicial = int(rng.integers(0, intervalo_dias))
            fecha_emision = FECHA_INICIO + timedelta(days=offset_inicial)
            while fecha_emision <= FECHA_FIN:
                fecha_venc = fecha_emision + timedelta(days=int(plan.plazo_cumplimiento_dias))
                filas_ordenes.append(dict(
                    orden_id=orden_id, plan_id=plan.plan_id, activo_id=act.activo_id,
                    fecha_emision=fecha_emision, fecha_vencimiento=fecha_venc,
                    fecha_ejecucion=None, estado="pendiente",
                ))
                orden_id += 1
                fecha_emision = fecha_emision + timedelta(days=intervalo_dias)

# --- órdenes ad-hoc: decisión de supervisión, sin plan detrás ---
N_ORDENES_ADHOC = 60
for _ in range(N_ORDENES_ADHOC):
    idx = rng.integers(0, len(activos_relevantes))
    act = activos_relevantes.iloc[idx]
    dias_random = int(rng.integers(0, (FECHA_FIN - FECHA_INICIO).days))
    fecha_emision = FECHA_INICIO + timedelta(days=dias_random)
    plazo = int(rng.choice([7, 15, 30]))
    filas_ordenes.append(dict(
        orden_id=orden_id, plan_id=None, activo_id=act.activo_id,
        fecha_emision=fecha_emision, fecha_vencimiento=fecha_emision + timedelta(days=plazo),
        fecha_ejecucion=None, estado="pendiente",
    ))
    orden_id += 1

ordenes = pd.DataFrame(filas_ordenes)
guardar(ordenes, "ordenes")
print(f"\nÓrdenes por plan (top 10):")
print(ordenes.groupby("plan_id", dropna=False).size().sort_values(ascending=False).head(10))
ordenes.head()

ordenes: 2889 filas, 7 columnas -> output_csv/ordenes.csv

Órdenes por plan (top 10):
plan_id
17.0    419
23.0    369
25.0    343
6.0     318
8.0     184
21.0    144
7.0     136
4.0     122
13.0    122
3.0     119
dtype: int64


,orden_id,plan_id,activo_id,fecha_emision,fecha_vencimiento,fecha_ejecucion,estado
0,1,1.0,104,2024-03-26,2024-04-25,None,pendiente
1,2,1.0,104,2025-03-26,2025-04-25,None,pendiente
2,3,1.0,121,2024-11-07,2024-12-07,None,pendiente
3,4,1.0,121,2025-11-07,2025-12-07,None,pendiente
4,5,1.0,122,2024-03-29,2024-04-28,None,pendiente


### 5.4 Verificación rápida

Chequeamos que todas las órdenes referencien activos y planes
válidos, y que la cantidad total sea razonable para 2 años de
historia.


In [15]:
ids_activos_validos = set(activos.activo_id)
ids_planes_validos = set(plan_mantenimiento_preventivo.plan_id)

activos_invalidos = ordenes[~ordenes.activo_id.isin(ids_activos_validos)]
planes_invalidos = ordenes[ordenes.plan_id.notna() & ~ordenes.plan_id.isin(ids_planes_validos)]

print(f"Total de órdenes: {len(ordenes)}")
print(f"Órdenes ad-hoc (sin plan): {ordenes.plan_id.isna().sum()}")
print(f"Órdenes con activo_id inválido: {len(activos_invalidos)} (debería ser 0)")
print(f"Órdenes con plan_id inválido: {len(planes_invalidos)} (debería ser 0)")

Total de órdenes: 2889
Órdenes ad-hoc (sin plan): 60
Órdenes con activo_id inválido: 0 (debería ser 0)
Órdenes con plan_id inválido: 0 (debería ser 0)


## 6. `eventos_mantenimiento`

Esta es la tabla más rica del dataset, porque de acá salen dos flujos
distintos que después se concatenan:

1. **Correctivos espontáneos** (sin orden): las fallas en sí. Se
   generan con un **proceso de renovación con tiempos entre fallas
   Weibull** por cada equipo — no aleatorio uniforme — para que el
   dataset tenga un patrón de confiabilidad real que se pueda
   analizar (curva de bañera, MTBF, etc.).
2. **Preventivos** (con orden): salen de expandir las `ordenes` que
   efectivamente se ejecutaron. Acá también resolvemos, recién ahora,
   el `estado` final y `fecha_ejecucion` de cada orden.

### 6.1 Por qué Weibull

La distribución de Weibull es el estándar en ingeniería de
confiabilidad para modelar tiempos hasta la falla:

- **forma (β) < 1** → tasa de falla decreciente (mortalidad infantil:
  fallas de fábrica/instalación, típico de equipos recién instalados)
- **β = 1** → tasa de falla constante (fallas aleatorias, es
  simplemente una exponencial)
- **β > 1** → tasa de falla creciente (desgaste: cuanto más viejo,
  más probable que falle)

Como en nuestro dataset la `fecha_alta` de los equipos coincide con la
puesta en marcha de la planta (años antes del período 2024-2025 que
estamos generando), la mayoría de los equipos ya está en zona de
**vida útil** o **desgaste** al arrancar la ventana de datos — no en
mortalidad infantil. Usamos la antigüedad de cada equipo al inicio del
período para elegir β, y la criticidad/categoría del activo para
definir el MTBF objetivo (y de ahí derivar el parámetro de escala η).


In [16]:
from scipy.stats import gamma as gamma_dist
import math

def beta_por_antiguedad(antiguedad_anios, rng):
    """Elige el parámetro de forma de Weibull según la antigüedad
    del equipo al inicio de la ventana de datos."""
    if antiguedad_anios < 2:
        return rng.uniform(0.6, 0.9)   # mortalidad infantil
    elif antiguedad_anios < 7:
        return rng.uniform(0.9, 1.3)   # fallas aleatorias (vida útil)
    else:
        return rng.uniform(1.5, 2.6)   # desgaste


def mtbf_objetivo_dias(criticidad, categoria, rng):
    """Tiempo medio entre fallas 'objetivo', en días, según
    criticidad (a mayor criticidad, más exigido, falla más seguido)
    y categoría del activo (algunas familias son más robustas)."""
    base = {"Alta": 150, "Media": 350, "Baja": 700}[criticidad]
    multiplicador = 1.0
    if "Eléctrico" in categoria:
        multiplicador *= 1.3
    if "Mecánico" in categoria:
        multiplicador *= 0.8
    if "Hidráulico" in categoria:
        multiplicador *= 0.9
    if "Combustión" in categoria:
        multiplicador *= 0.7
    if "Electrónico" in categoria:
        multiplicador *= 1.15
    # ruido propio del activo individual (no todos los motores son iguales)
    multiplicador *= rng.uniform(0.75, 1.25)
    return base * multiplicador


def generar_fallas_weibull(activo_id, fecha_alta, criticidad, categoria, rng):
    """
    Genera las fechas de falla (correctivos espontáneos) de un equipo
    dentro de la ventana [FECHA_INICIO, FECHA_FIN], usando un proceso
    de renovación: cada intervalo entre fallas se sortea de una
    Weibull(beta, eta) independiente.
    """
    antiguedad_anios = (FECHA_INICIO - fecha_alta).days / 365.25
    antiguedad_anios = max(antiguedad_anios, 0)

    beta = beta_por_antiguedad(antiguedad_anios, rng)
    mtbf = mtbf_objetivo_dias(criticidad, categoria, rng)
    # eta (escala) tal que la media de la Weibull coincida con el MTBF objetivo
    eta = mtbf / gamma_dist.mean(a=1) ** 0  # placeholder, se corrige abajo
    eta = mtbf / math.gamma(1 + 1 / beta)

    fallas = []
    # offset de "fase" aleatorio: no todos los equipos arrancan su
    # reloj de desgaste el mismo día del período
    t = FECHA_INICIO - timedelta(days=int(rng.uniform(0, eta)))
    while t <= FECHA_FIN:
        intervalo_dias = rng.weibull(beta) * eta
        t = t + timedelta(days=float(intervalo_dias))
        if FECHA_INICIO <= t <= FECHA_FIN:
            fallas.append(t)
    return fallas


print("Funciones de generación Weibull definidas.")

Funciones de generación Weibull definidas.


### 6.2 Tipos de falla y duración según categoría del activo

Mapeamos qué tipos de falla son típicos de cada categoría (un
transformador no sufre "obstrucción", una bomba no sufre
"cortocircuito"), y definimos rangos de duración de la intervención
según la criticidad del activo.


In [17]:
# (tipo_falla, peso) por categoría — pesos relativos, no necesitan sumar 1
TIPO_FALLA_POR_CATEGORIA = {
    "Eléctrico":    [("Cortocircuito", 3), ("Sobrecalentamiento", 3),
                      ("Falla de aislación", 2), ("Falla de rodamiento", 1)],
    "Electrónico":  [("Error de programación/control", 3), ("Sobrecalentamiento", 2),
                      ("Cortocircuito", 1)],
    "Mecánico":     [("Desgaste mecánico", 3), ("Vibración excesiva", 2),
                      ("Rotura estructural", 1), ("Falla de lubricación", 2),
                      ("Falla de rodamiento", 2), ("Obstrucción / atascamiento", 1)],
    "Hidráulico":   [("Fuga de fluido", 3), ("Falla de lubricación", 1),
                      ("Obstrucción / atascamiento", 1), ("Vibración excesiva", 1)],
    "Combustión":   [("Sobrecalentamiento", 2), ("Falla de lubricación", 2),
                      ("Corrosión", 1)],
}
FALLAS_GENERICAS = [("Corrosión", 1), ("Fin de vida útil", 1)]

# probabilidad de que un correctivo quede SIN tipo_falla_id registrado
# (dato faltante real, no confundir con el NULL de los preventivos)
PROB_FALLA_SIN_REGISTRAR = 0.08


def elegir_tipo_falla(categoria, rng):
    candidatos = list(FALLAS_GENERICAS)
    for clave, lista in TIPO_FALLA_POR_CATEGORIA.items():
        if clave in categoria:
            candidatos += lista
    nombres = [c[0] for c in candidatos]
    pesos = np.array([c[1] for c in candidatos], dtype=float)
    pesos = pesos / pesos.sum()
    elegido = rng.choice(nombres, p=pesos)
    return TF[elegido]


def duracion_horas_correctivo(criticidad, rng):
    rango = {"Alta": (4, 48), "Media": (2, 24), "Baja": (1, 12)}[criticidad]
    return rng.uniform(*rango)


def prob_paro_planta_correctivo(criticidad):
    return {"Alta": 0.60, "Media": 0.25, "Baja": 0.05}[criticidad]


DESCRIPCIONES_CORRECTIVO = [
    "falla detectada por operador, se interviene", "para por alarma",
    "reparación de urgencia", "PARADA IMPREVISTA", "falla en arranque",
    "detectado ruido anormal", "pérdida reportada por turno noche",
    "s/observaciones", "intervención según diagnóstico previo",
]

print("Catálogos auxiliares de fallas listos.")

Catálogos auxiliares de fallas listos.


### 6.3 Generar correctivos espontáneos (Weibull) para cada equipo

Recorremos todos los activos de nivel 2 y 3 (los equipos reales, no
las áreas) y les generamos su propia secuencia de fallas.


In [18]:
filas_eventos = []
evento_id = 1

activos_relevantes = activos_relevantes.copy()  # ya filtrado a nivel 2 y 3
activos_relevantes["fecha_alta"] = pd.to_datetime(activos_relevantes["fecha_alta"]).dt.date

for _, act in activos_relevantes.iterrows():
    categoria = tipo_activo.loc[tipo_activo.tipo_activo_id == act.tipo_activo_id, "categoria"].iloc[0]
    fallas = generar_fallas_weibull(act.activo_id, act.fecha_alta, act.criticidad, categoria, rng)

    for fecha_falla in fallas:
        hora_inicio = datetime.combine(fecha_falla, datetime.min.time()) + timedelta(
            hours=float(rng.uniform(0, 24))
        )
        duracion = duracion_horas_correctivo(act.criticidad, rng)
        hora_fin = hora_inicio + timedelta(hours=float(duracion))

        tipo_falla_id = elegir_tipo_falla(categoria, rng)
        if rng.random() < PROB_FALLA_SIN_REGISTRAR:
            tipo_falla_id = None  # dato faltante real

        filas_eventos.append(dict(
            evento_id=evento_id, activo_id=act.activo_id, orden_id=None,
            tipo_mantenimiento="correctivo", tipo_falla_id=tipo_falla_id,
            fecha_hora_inicio=hora_inicio, fecha_hora_fin=hora_fin,
            genero_paro_planta=bool(rng.random() < prob_paro_planta_correctivo(act.criticidad)),
            descripcion=rng.choice(DESCRIPCIONES_CORRECTIVO),
        ))
        evento_id += 1

print(f"Correctivos generados: {evento_id - 1}")

Correctivos generados: 319


### 6.4 Resolver las órdenes preventivas y generar sus eventos

Ahora sí cerramos el círculo con `ordenes`: decidimos, para cada
orden, si se ejecutó a tiempo, tarde, si fue anulada, o si quedó
vencida sin ejecutar. Solo las ejecutadas generan un evento en
`eventos_mantenimiento`.

Un matiz de diseño: si una orden se ejecuta **después** de su
vencimiento, la marcamos igual como `vencida` (incumplió el plazo),
aunque el trabajo se haya hecho — mantenemos `fecha_ejecucion`
completa en ese caso. Es una decisión discutible pero razonable, y la
dejamos anotada para que la tengas presente en el análisis.


In [19]:
DESCRIPCIONES_PREVENTIVO = [
    "tarea de rutina según plan", "OK sin observaciones", "cumplido",
    "se realizó tarea programada", "rutina realizada, todo en orden",
    "pendiente revisión adicional", "s/n",
]

TF_PREVENTIVO = TF["Mantenimiento programado (sin falla)"]

ordenes = ordenes.copy()
ordenes["fecha_emision"] = pd.to_datetime(ordenes["fecha_emision"]).dt.date
ordenes["fecha_vencimiento"] = pd.to_datetime(ordenes["fecha_vencimiento"]).dt.date

nuevas_filas_ordenes = []

for _, ord_ in ordenes.iterrows():
    r = rng.random()
    act_row = activos.loc[activos.activo_id == ord_.activo_id].iloc[0]

    if r < 0.82:
        # se ejecuta -> a tiempo u tarde
        if rng.random() < 0.85:
            dias_para_ejecutar = int(rng.integers(0, max(1, (ord_.fecha_vencimiento - ord_.fecha_emision).days)))
            fecha_ejecucion = ord_.fecha_emision + timedelta(days=dias_para_ejecutar)
            estado = "cumplida"
        else:
            atraso = int(rng.integers(1, 20))
            fecha_ejecucion = ord_.fecha_vencimiento + timedelta(days=atraso)
            estado = "vencida"  # se hizo, pero tarde: igual queda vencida

        if fecha_ejecucion <= FECHA_FIN:
            hora_inicio = datetime.combine(fecha_ejecucion, datetime.min.time()) + timedelta(
                hours=float(rng.uniform(6, 18))  # trabajo preventivo en horario diurno
            )
            duracion = rng.uniform(6, 24) if pd.notna(ord_.plan_id) and \
                       "parada" in str(plan_mantenimiento_preventivo.loc[
                           plan_mantenimiento_preventivo.plan_id == ord_.plan_id, "descripcion_tarea"
                       ].values).lower() else rng.uniform(1, 6)
            hora_fin = hora_inicio + timedelta(hours=float(duracion))

            tipo_falla_id = TF_PREVENTIVO
            if rng.random() < 0.05:  # rara vez, tampoco se registra en preventivos
                tipo_falla_id = None

            filas_eventos.append(dict(
                evento_id=evento_id, activo_id=ord_.activo_id, orden_id=ord_.orden_id,
                tipo_mantenimiento="preventivo", tipo_falla_id=tipo_falla_id,
                fecha_hora_inicio=hora_inicio, fecha_hora_fin=hora_fin,
                genero_paro_planta=bool(rng.random() < 0.08),
                descripcion=rng.choice(DESCRIPCIONES_PREVENTIVO),
            ))
            evento_id += 1
        else:
            # la ejecución hubiera caído fuera de la ventana de datos
            fecha_ejecucion = None
            estado = "pendiente"
    else:
        # no se ejecutó
        if rng.random() < 0.12:
            estado = "anulada"
            fecha_ejecucion = None
        elif ord_.fecha_vencimiento <= FECHA_FIN:
            estado = "vencida"
            fecha_ejecucion = None
        else:
            estado = "pendiente"
            fecha_ejecucion = None

    nuevas_filas_ordenes.append(dict(fecha_ejecucion=fecha_ejecucion, estado=estado))

ordenes["fecha_ejecucion"] = [f["fecha_ejecucion"] for f in nuevas_filas_ordenes]
ordenes["estado"] = [f["estado"] for f in nuevas_filas_ordenes]

print("Distribución final de estado de órdenes:")
print(ordenes.estado.value_counts())
print()
print(f"Total de eventos tras sumar preventivos: {evento_id - 1}")

Distribución final de estado de órdenes:
estado
cumplida     1987
vencida       820
anulada        54
pendiente      28
Name: count, dtype: int64

Total de eventos tras sumar preventivos: 2673


### 6.5 Armar `eventos_mantenimiento` final y guardarlo


In [20]:
eventos_mantenimiento = pd.DataFrame(filas_eventos)
guardar(eventos_mantenimiento, "eventos_mantenimiento")
guardar(ordenes, "ordenes")  # la reescribimos con estado/fecha_ejecucion definitivos

print()
print("Eventos por tipo de mantenimiento:")
print(eventos_mantenimiento.tipo_mantenimiento.value_counts())
print()
print("Eventos correctivos sin tipo_falla_id (dato faltante):",
      eventos_mantenimiento[(eventos_mantenimiento.tipo_mantenimiento == "correctivo") &
                             (eventos_mantenimiento.tipo_falla_id.isna())].shape[0])
eventos_mantenimiento.head()

eventos_mantenimiento: 2673 filas, 9 columnas -> output_csv/eventos_mantenimiento.csv
ordenes: 2889 filas, 7 columnas -> output_csv/ordenes.csv

Eventos por tipo de mantenimiento:
tipo_mantenimiento
preventivo    2354
correctivo     319
Name: count, dtype: int64

Eventos correctivos sin tipo_falla_id (dato faltante): 23


,evento_id,activo_id,orden_id,tipo_mantenimiento,tipo_falla_id,fecha_hora_inicio,fecha_hora_fin,genero_paro_planta,descripcion
0,1,2,NaN,correctivo,10.0,2024-04-09 09:03:05.300383,2024-04-10 02:52:34.853051,False,PARADA IMPREVISTA
1,2,2,NaN,correctivo,12.0,2024-07-12 13:56:37.481892,2024-07-13 00:56:41.368120,False,detectado ruido anormal
2,3,2,NaN,correctivo,5.0,2024-12-28 04:43:18.524417,2024-12-28 15:01:13.238261,True,reparación de urgencia
3,4,2,NaN,correctivo,1.0,2025-03-30 08:08:50.877964,2025-03-31 07:09:18.830599,False,pérdida reportada por turno noche
4,5,2,NaN,correctivo,1.0,2025-04-17 21:08:41.514886,2025-04-18 05:15:48.092750,False,"falla detectada por operador, se interviene"


### 6.6 Verificación rápida

Chequeamos integridad referencial y que la duración de los eventos
sea siempre positiva (fin después de inicio).


In [21]:
ids_activos_validos = set(activos.activo_id)
ids_ordenes_validas = set(ordenes.orden_id)

activo_invalido = eventos_mantenimiento[~eventos_mantenimiento.activo_id.isin(ids_activos_validos)]
orden_invalida = eventos_mantenimiento[
    eventos_mantenimiento.orden_id.notna() & ~eventos_mantenimiento.orden_id.isin(ids_ordenes_validas)
]
duracion_negativa = eventos_mantenimiento[
    pd.to_datetime(eventos_mantenimiento.fecha_hora_fin) <= pd.to_datetime(eventos_mantenimiento.fecha_hora_inicio)
]

print(f"Total eventos: {len(eventos_mantenimiento)}")
print(f"Eventos con activo_id inválido: {len(activo_invalido)} (debería ser 0)")
print(f"Eventos con orden_id inválido: {len(orden_invalida)} (debería ser 0)")
print(f"Eventos con duración <= 0: {len(duracion_negativa)} (debería ser 0)")

Total eventos: 2673
Eventos con activo_id inválido: 0 (debería ser 0)
Eventos con orden_id inválido: 0 (debería ser 0)
Eventos con duración <= 0: 0 (debería ser 0)


## 7. `evento_tecnico` y `evento_repuesto`

### 7.1 `evento_tecnico`

Para cada evento, asignamos entre 1 y 3 técnicos **de la misma
planta** que el activo intervenido (recordemos: técnicos fijos por
planta). Le damos más probabilidad a los técnicos cuya especialidad
coincide con la categoría del activo, sin hacerlo determinístico —
también puede tocarle a alguien de otra especialidad, como pasa en la
vida real cuando falta personal específico disponible.

No guardamos `horas_trabajadas`: la decisión fue asumir que todos los
técnicos asignados a un evento trabajan la duración completa del
evento (horas-hombre = duración × cantidad de técnicos), calculable en
el análisis.


In [22]:
# técnicos agrupados por planta para no filtrar el DataFrame en cada iteración
tecnicos_por_planta = {pid: grp for pid, grp in tecnicos.groupby("planta_id")}

# mapeo simple especialidad <-> categoría, para pesar la selección
AFINIDAD_ESPECIALIDAD = {
    "Eléctrico":            "Eléctrico",
    "Electrónico/Control":  "Electrónico",
    "Mecánico":             "Mecánico",
    "Hidráulico":           "Hidráulico",
    "General":              None,  # el técnico general no tiene afinidad particular
}

filas_evento_tecnico = []

# activo_id -> planta_id y -> categoria, para lookup rápido
activo_a_planta = dict(zip(activos.activo_id, activos.planta_id))
activo_a_tipo = dict(zip(activos.activo_id, activos.tipo_activo_id))
tipo_a_categoria = dict(zip(tipo_activo.tipo_activo_id, tipo_activo.categoria))

for _, ev in eventos_mantenimiento.iterrows():
    planta_id = activo_a_planta[ev.activo_id]
    categoria = tipo_a_categoria[activo_a_tipo[ev.activo_id]]
    disponibles = tecnicos_por_planta[planta_id]

    pesos = disponibles.especialidad.map(
        lambda e: 3.0 if (AFINIDAD_ESPECIALIDAD.get(e) and AFINIDAD_ESPECIALIDAD[e] in categoria) else 1.0
    ).values
    pesos = pesos / pesos.sum()

    n_tecnicos = int(rng.choice([1, 2, 3], p=[0.55, 0.35, 0.10]))
    n_tecnicos = min(n_tecnicos, len(disponibles))
    elegidos = rng.choice(disponibles.tecnico_id.values, size=n_tecnicos, replace=False, p=pesos)

    for tecnico_id in elegidos:
        filas_evento_tecnico.append(dict(evento_id=ev.evento_id, tecnico_id=int(tecnico_id)))

evento_tecnico = pd.DataFrame(filas_evento_tecnico)
guardar(evento_tecnico, "evento_tecnico")
print(f"Promedio de técnicos por evento: {evento_tecnico.groupby('evento_id').size().mean():.2f}")
evento_tecnico.head()

evento_tecnico: 4143 filas, 2 columnas -> output_csv/evento_tecnico.csv
Promedio de técnicos por evento: 1.55


,evento_id,tecnico_id
0,1,2
1,2,1
2,3,4
3,3,5
4,4,5


### 7.2 `evento_repuesto`

Solo una parte de los eventos consume repuestos/insumos (una
inspección de rutina puede no usar nada; una reparación mayor sí). El
costo se guarda **al momento del evento**, no el de catálogo — y para
que la inflación argentina quede reflejada de forma realista, lo
escalamos usando la trayectoria de `tipo_cambio`: asumimos que el
costo de catálogo (`costo_unitario`) está expresado a valores del
**último mes del período**, y para eventos anteriores lo
"retrocedemos" en proporción a cuánto valía el dólar en ese momento
(más un ruido para no ser mecánico).


In [23]:
tipo_cambio_por_periodo = dict(zip(
    pd.to_datetime(tipo_cambio.periodo).dt.to_period("M"),
    tipo_cambio.valor_promedio,
))
tc_referencia = tipo_cambio.valor_promedio.iloc[-1]  # último mes = referencia "actual"

repuesto_id_to_costo = dict(zip(repuestos_insumos.repuesto_id, repuestos_insumos.costo_unitario))
repuesto_ids = repuestos_insumos.repuesto_id.values

# probabilidad de usar repuestos, y cantidad, según tipo de mantenimiento
PROB_USA_REPUESTO = {"correctivo": 0.75, "preventivo": 0.40}

filas_evento_repuesto = []

for _, ev in eventos_mantenimiento.iterrows():
    if rng.random() > PROB_USA_REPUESTO[ev.tipo_mantenimiento]:
        continue

    n_items = int(rng.choice([1, 2, 3], p=[0.6, 0.3, 0.1]))
    items = rng.choice(repuesto_ids, size=n_items, replace=False)

    periodo_evento = pd.Period(pd.Timestamp(ev.fecha_hora_inicio), freq="M")
    tc_evento = tipo_cambio_por_periodo.get(periodo_evento, tc_referencia)
    factor_historico = (tc_evento / tc_referencia) * rng.uniform(0.9, 1.1)

    for repuesto_id in items:
        cantidad = round(float(rng.uniform(1, 5)), 1)
        costo_hist = repuesto_id_to_costo[repuesto_id] * factor_historico
        filas_evento_repuesto.append(dict(
            evento_id=ev.evento_id, repuesto_id=int(repuesto_id),
            cantidad_utilizada=cantidad,
            costo_unitario_momento=round(costo_hist, 2),
        ))

evento_repuesto = pd.DataFrame(filas_evento_repuesto)
guardar(evento_repuesto, "evento_repuesto")
evento_repuesto.head()

evento_repuesto: 1737 filas, 4 columnas -> output_csv/evento_repuesto.csv


,evento_id,repuesto_id,cantidad_utilizada,costo_unitario_momento
0,1,1,3.7,22527.66
1,2,6,1.9,4182.19
2,3,15,4.2,1624.27
3,5,5,2.0,21589.07
4,5,14,5.0,37106.22


### 7.3 Verificación rápida

In [24]:
ids_eventos_validos = set(eventos_mantenimiento.evento_id)
ids_tecnicos_validos = set(tecnicos.tecnico_id)
ids_repuestos_validos = set(repuestos_insumos.repuesto_id)

et_evento_inv = evento_tecnico[~evento_tecnico.evento_id.isin(ids_eventos_validos)]
et_tecnico_inv = evento_tecnico[~evento_tecnico.tecnico_id.isin(ids_tecnicos_validos)]
er_evento_inv = evento_repuesto[~evento_repuesto.evento_id.isin(ids_eventos_validos)]
er_repuesto_inv = evento_repuesto[~evento_repuesto.repuesto_id.isin(ids_repuestos_validos)]

print(f"evento_tecnico: {len(evento_tecnico)} filas | evento_id inválidos: {len(et_evento_inv)} | tecnico_id inválidos: {len(et_tecnico_inv)}")
print(f"evento_repuesto: {len(evento_repuesto)} filas | evento_id inválidos: {len(er_evento_inv)} | repuesto_id inválidos: {len(er_repuesto_inv)}")
print()
print(f"% de eventos con al menos un repuesto: {evento_repuesto.evento_id.nunique() / len(eventos_mantenimiento):.1%}")

evento_tecnico: 4143 filas | evento_id inválidos: 0 | tecnico_id inválidos: 0
evento_repuesto: 1737 filas | evento_id inválidos: 0 | repuesto_id inválidos: 0

% de eventos con al menos un repuesto: 44.3%


## 8. `mediciones` — horas de uso acumuladas

Generamos lecturas de horómetro para cada equipo (nivel 2 y 3), con
una frecuencia de registro mensual (simulando que alguien pasa una
vez al mes a anotar el horómetro — no es un sensor continuo, es
consistente con el resto del dataset donde la carga es mayormente
manual).

**Punto importante de coherencia**: las horas acumuladas tienen que
respetar la tasa de uso implícita que ya usamos para derivar la
cadencia de las órdenes `por_horas_uso` (`HORAS_USO_PROMEDIO_DIA =
16`). Le sumamos variabilidad por activo (algunos equipos trabajan más
que otros) y ruido mes a mes, pero manteniendo esa media global para
que, si más adelante cruzás horas acumuladas con antigüedad de
componentes, el orden de magnitud tenga sentido.


In [25]:
filas_mediciones = []
medicion_id = 1

TM_HORAS = TM["Horas de uso acumuladas"]

for _, act in activos_relevantes.iterrows():
    # tasa de uso propia del equipo, con variabilidad alrededor del
    # promedio global (algunos equipos operan casi 24hs, otros menos)
    tasa_propia = HORAS_USO_PROMEDIO_DIA * rng.uniform(0.5, 1.3)

    acumulado = 0.0
    fecha_lectura = FECHA_INICIO.replace(day=1)
    dia_anterior = FECHA_INICIO

    while fecha_lectura <= FECHA_FIN:
        dias_transcurridos = (fecha_lectura - dia_anterior).days
        # ruido mensual: algunos meses se trabaja más/menos (mantenimiento,
        # baja demanda, etc.)
        horas_del_periodo = max(0.0, dias_transcurridos * tasa_propia * rng.uniform(0.7, 1.15))
        acumulado += horas_del_periodo

        filas_mediciones.append(dict(
            medicion_id=medicion_id, activo_id=act.activo_id,
            tipo_medicion_id=TM_HORAS,
            fecha_hora=datetime.combine(fecha_lectura, datetime.min.time()) + timedelta(hours=8),
            valor=round(acumulado, 1),
        ))
        medicion_id += 1

        dia_anterior = fecha_lectura
        # avanzar al primer día del mes siguiente
        if fecha_lectura.month == 12:
            fecha_lectura = fecha_lectura.replace(year=fecha_lectura.year + 1, month=1)
        else:
            fecha_lectura = fecha_lectura.replace(month=fecha_lectura.month + 1)

mediciones = pd.DataFrame(filas_mediciones)
guardar(mediciones, "mediciones")
print(f"Lecturas por activo (promedio): {mediciones.groupby('activo_id').size().mean():.1f}")
mediciones.head()

mediciones: 2928 filas, 5 columnas -> output_csv/mediciones.csv
Lecturas por activo (promedio): 24.0


,medicion_id,activo_id,tipo_medicion_id,fecha_hora,valor
0,1,2,1,2024-01-01 08:00:00,0.0
1,2,2,1,2024-02-01 08:00:00,384.3
2,3,2,1,2024-03-01 08:00:00,693.7
3,4,2,1,2024-04-01 08:00:00,1055.3
4,5,2,1,2024-05-01 08:00:00,1380.9


### 8.1 Verificación rápida

Chequeamos que el acumulado de horas nunca decrezca dentro del mismo
activo (un horómetro real es monótono creciente) y que no haya
activo_id inválidos.


In [26]:
ids_activos_validos = set(activos.activo_id)
med_invalidas = mediciones[~mediciones.activo_id.isin(ids_activos_validos)]

no_monotono = 0
for activo_id, grupo in mediciones.sort_values("fecha_hora").groupby("activo_id"):
    if not grupo.valor.is_monotonic_increasing:
        no_monotono += 1

print(f"Total mediciones: {len(mediciones)}")
print(f"Mediciones con activo_id inválido: {len(med_invalidas)} (debería ser 0)")
print(f"Activos con horómetro no monótono creciente: {no_monotono} (debería ser 0)")

Total mediciones: 2928
Mediciones con activo_id inválido: 0 (debería ser 0)
Activos con horómetro no monótono creciente: 0 (debería ser 0)


## 9. `log_operativo` — bitácora ruidosa de eventos de planta

Como acordamos, esto simula un log crudo tipo SCADA/historian: mezcla
eventos de planta entera (`activo_id` nulo, ej. corte de suministro
externo) con eventos de un activo puntual (ej. drenaje de una línea),
y usa el catálogo `tipo_evento_operativo` ya "sucio" a propósito
(mayúsculas inconsistentes, abreviaturas, variantes redundantes).

Generamos dos fuentes de registros:

1. **Ruido operativo de fondo**: eventos frecuentes y menores
   (cambios de turno, purgas, drenajes, alarmas menores) a lo largo
   de todo el período, independientes de fallas.
2. **Paradas/arranques asociados a eventos de mantenimiento**: cuando
   un evento de `eventos_mantenimiento` generó paro de planta
   (`genero_paro_planta = True`), agregamos un par parada→arranque en
   el log con fechas cercanas al evento (no exactamente iguales, para
   simular el desfasaje real entre "se rompió" y "se registró el
   corte en el sistema").


In [27]:
# clasificar los tipos de evento operativo "sucios" en sus categorías reales,
# para poder generarlos con criterio (aunque el analista después tenga que
# re-descubrir esta clasificación limpiando el texto)
CATEGORIAS_EVENTO_OP = {
    "parada":    ["PARADA", "parada_planta", "STOP", "Parada por falla", "parada prog."],
    "arranque":  ["ARRANQUE", "arranque_planta", "START", "puesta en marcha"],
    "rutina":    ["drenaje", "DRENAJE LN2", "purga", "PURGA VALVULA", "limpieza filtros"],
    "alarma":    ["ALARMA TEMP", "disparo_termico", "proteccion activada", "ALARM_HIGH_PRESS"],
    "turno":     ["cambio_turno", "CAMBIO DE TURNO", "ajuste_setpoint", "cambio de producto"],
    "externo":   ["corte_suministro", "CORTE ELECTRICO EXTERNO", "falta de agua", "corte gas"],
    "calidad":   ["producto_fuera_especificacion", "PARO POR CALIDAD", "merma"],
}

filas_log = []
log_id = 1

# --- 1) ruido operativo de fondo, independiente de fallas ---
dias_totales = (FECHA_FIN - FECHA_INICIO).days
for _, planta in plantas.iterrows():
    # cambios de turno: 3 por día, todos los días, en TODAS las plantas
    for d in range(0, dias_totales + 1, 1):
        if rng.random() < 0.98:  # casi siempre se registra, a veces no
            fecha = FECHA_INICIO + timedelta(days=d)
            for hora_turno in [6, 14, 22]:
                if rng.random() < 0.5:  # no todos los cambios de turno quedan logueados
                    filas_log.append(dict(
                        log_id=log_id, planta_id=planta.planta_id, activo_id=None,
                        fecha_hora=datetime.combine(fecha, datetime.min.time()) + timedelta(hours=hora_turno),
                        tipo_evento_id=TEO[rng.choice(CATEGORIAS_EVENTO_OP["turno"])],
                        descripcion=rng.choice(["", "s/n", "ok", "normal"]),
                    ))
                    log_id += 1

    # eventos de rutina (drenajes, purgas) sobre activos puntuales de esa planta
    activos_de_planta = activos_relevantes[activos_relevantes.planta_id == planta.planta_id]
    n_eventos_rutina = int(dias_totales * 0.8)  # bastante frecuentes
    for _ in range(n_eventos_rutina):
        act = activos_de_planta.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
        dia = FECHA_INICIO + timedelta(days=int(rng.integers(0, dias_totales)))
        hora = float(rng.uniform(0, 24))
        filas_log.append(dict(
            log_id=log_id, planta_id=planta.planta_id, activo_id=act.activo_id,
            fecha_hora=datetime.combine(dia, datetime.min.time()) + timedelta(hours=hora),
            tipo_evento_id=TEO[rng.choice(CATEGORIAS_EVENTO_OP["rutina"])],
            descripcion=rng.choice(["OK", "ok.", "sin novedad", "", "DRN OK", "purga completa"]),
        ))
        log_id += 1

    # alarmas menores que no llegan a generar un evento de mantenimiento
    n_alarmas = int(dias_totales * 0.3)
    for _ in range(n_alarmas):
        act = activos_de_planta.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
        dia = FECHA_INICIO + timedelta(days=int(rng.integers(0, dias_totales)))
        hora = float(rng.uniform(0, 24))
        filas_log.append(dict(
            log_id=log_id, planta_id=planta.planta_id, activo_id=act.activo_id,
            fecha_hora=datetime.combine(dia, datetime.min.time()) + timedelta(hours=hora),
            tipo_evento_id=TEO[rng.choice(CATEGORIAS_EVENTO_OP["alarma"])],
            descripcion=rng.choice(["restablecida sola", "operador reseteo", "sin intervencion"]),
        ))
        log_id += 1

    # cortes externos y problemas de calidad: mucho más esporádicos
    for categoria, n_eventos in [("externo", int(dias_totales * 0.03)), ("calidad", int(dias_totales * 0.05))]:
        for _ in range(n_eventos):
            dia = FECHA_INICIO + timedelta(days=int(rng.integers(0, dias_totales)))
            hora = float(rng.uniform(0, 24))
            filas_log.append(dict(
                log_id=log_id, planta_id=planta.planta_id, activo_id=None,
                fecha_hora=datetime.combine(dia, datetime.min.time()) + timedelta(hours=hora),
                tipo_evento_id=TEO[rng.choice(CATEGORIAS_EVENTO_OP[categoria])],
                descripcion=rng.choice(["", "ver informe adjunto", "aviso proveedor externo"]),
            ))
            log_id += 1

print(f"Registros de ruido operativo de fondo: {log_id - 1}")

Registros de ruido operativo de fondo: 9657


In [28]:
# --- 2) paradas/arranques asociados a eventos que generaron paro de planta ---
eventos_con_paro = eventos_mantenimiento[eventos_mantenimiento.genero_paro_planta]

for _, ev in eventos_con_paro.iterrows():
    planta_id = activo_a_planta[ev.activo_id]

    # la parada se registra en el log un rato ANTES o coincidiendo con el
    # inicio del evento (el sistema corta antes de que llegue el técnico)
    desfasaje_parada = timedelta(minutes=float(rng.uniform(-30, 15)))
    hora_parada = pd.Timestamp(ev.fecha_hora_inicio) + desfasaje_parada

    filas_log.append(dict(
        log_id=log_id, planta_id=planta_id, activo_id=ev.activo_id,
        fecha_hora=hora_parada,
        tipo_evento_id=TEO[rng.choice(CATEGORIAS_EVENTO_OP["parada"])],
        descripcion=rng.choice(["por mantenimiento", "PARADA POR FALLA", "", "ver OT"]),
    ))
    log_id += 1

    # el arranque se registra un rato DESPUES del fin del evento
    # (el equipo no vuelve a producir instantáneamente)
    desfasaje_arranque = timedelta(minutes=float(rng.uniform(5, 90)))
    hora_arranque = pd.Timestamp(ev.fecha_hora_fin) + desfasaje_arranque

    filas_log.append(dict(
        log_id=log_id, planta_id=planta_id, activo_id=ev.activo_id,
        fecha_hora=hora_arranque,
        tipo_evento_id=TEO[rng.choice(CATEGORIAS_EVENTO_OP["arranque"])],
        descripcion=rng.choice(["reanuda produccion", "OK", "arranque normal", ""]),
    ))
    log_id += 1

log_operativo = pd.DataFrame(filas_log).sort_values("fecha_hora").reset_index(drop=True)
log_operativo["log_id"] = range(1, len(log_operativo) + 1)  # re-numerar en orden cronológico

guardar(log_operativo, "log_operativo")
print(f"Total log_operativo: {len(log_operativo)}")
log_operativo.head(10)

log_operativo: 10349 filas, 6 columnas -> output_csv/log_operativo.csv
Total log_operativo: 10349


,log_id,planta_id,activo_id,fecha_hora,tipo_evento_id,descripcion
0,1,4,109.0,2024-01-01 02:22:30.553267,13,sin novedad
1,2,1,NaN,2024-01-01 06:00:00.000000,20,normal
2,3,5,NaN,2024-01-01 06:00:00.000000,21,ok
3,4,3,63.0,2024-01-01 12:16:51.742203,11,ok.
4,5,1,NaN,2024-01-01 14:00:00.000000,20,ok
5,6,3,NaN,2024-01-01 14:00:00.000000,20,s/n
6,7,1,36.0,2024-01-01 15:05:13.224113,16,restablecida sola
7,8,2,NaN,2024-01-01 22:00:00.000000,20,normal
8,9,1,5.0,2024-01-01 22:58:37.897810,11,DRN OK
9,10,4,NaN,2024-01-02 06:00:00.000000,22,s/n


### 9.1 Verificación rápida

In [29]:
ids_plantas_validas = set(plantas.planta_id)
ids_activos_validos = set(activos.activo_id)
ids_tipo_evento_validos = set(tipo_evento_operativo.tipo_evento_id)

planta_inv = log_operativo[~log_operativo.planta_id.isin(ids_plantas_validas)]
activo_inv = log_operativo[log_operativo.activo_id.notna() & ~log_operativo.activo_id.isin(ids_activos_validos)]
tipo_inv = log_operativo[~log_operativo.tipo_evento_id.isin(ids_tipo_evento_validos)]

print(f"Total log_operativo: {len(log_operativo)}")
print(f"planta_id inválidos: {len(planta_inv)} (debería ser 0)")
print(f"activo_id inválidos: {len(activo_inv)} (debería ser 0)")
print(f"tipo_evento_id inválidos: {len(tipo_inv)} (debería ser 0)")
print()
print("% de registros con activo_id nulo (eventos de planta entera):",
      f"{log_operativo.activo_id.isna().mean():.1%}")

Total log_operativo: 10349
planta_id inválidos: 0 (debería ser 0)
activo_id inválidos: 0 (debería ser 0)
tipo_evento_id inválidos: 0 (debería ser 0)

% de registros con activo_id nulo (eventos de planta entera): 54.5%


## 10. Resumen final del dataset

Con esto quedan generadas las 16 tablas del modelo. Repaso de
volumen y un cierre de integridad cruzada antes de dar por terminada
la generación.


In [30]:
import glob

print("=" * 60)
print("RESUMEN FINAL DEL DATASET GENERADO")
print("=" * 60)
for path in sorted(glob.glob(os.path.join(OUT_DIR, "*.csv"))):
    df_tmp = pd.read_csv(path)
    nombre = os.path.basename(path).replace(".csv", "")
    print(f"{nombre:35s} {df_tmp.shape[0]:>7,} filas  {df_tmp.shape[1]:>3} columnas")
print("=" * 60)
print(f"Período cubierto: {FECHA_INICIO} a {FECHA_FIN}")
print(f"Semilla de aleatoriedad: 42 (reproducible)")

RESUMEN FINAL DEL DATASET GENERADO
activos                                 146 filas   13 columnas


evento_repuesto                       1,737 filas    4 columnas
evento_tecnico                        4,143 filas    2 columnas
eventos_mantenimiento                 2,673 filas    9 columnas
log_operativo                        10,349 filas    6 columnas
mediciones                            2,928 filas    5 columnas
ordenes                               2,889 filas    7 columnas
plan_mantenimiento_preventivo            25 filas    6 columnas
plantas                                   5 filas    5 columnas
repuestos_insumos                        15 filas    5 columnas
tecnicos                                 37 filas    6 columnas
tipo_activo                              20 filas    3 columnas
tipo_cambio                              24 filas    3 columnas
tipo_evento_operativo                    29 filas    2 columnas
tipo_falla                               14 filas    2 columnas
tipo_medicion                             1 filas    2 columnas
Período cubierto: 2024-01-01 a 2025-12-3